In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

## Imports

In [0]:

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

from repository.spark_repo import SparkRepository
from schemas.input_schema import (
    NEW_YORK_WEATHER_SCHEMA
)
from transformations.data_cleaning import (
    quarantine_null_values,
    quarantine_negative_values,
    quarantine_date_range
)
from utils.constants import (
    DATE_FORMAT
)

## Local Constants

In [0]:
# Range constants
_MIN_DATE = "2026-01-01"
_MAX_DATE = "2026-01-31" # Is included in range expr

In [0]:
# Load Input Data

## Load Taxi Data

In [0]:
taxi_raw_data = SparkRepository(spark).read(
    path="/Volumes/dbw_schwarz_test/default/temp/nytaxi/yellow_tripdata_2026-01.parquet",
    file_format="parquet")
print(f"Number of rows in taxi_raw_data: {taxi_raw_data.count()}")
display(taxi_raw_data.limit(10))

### Clean input data

cleaning taxi data

In [0]:
taxi_wo_null, taxi_q_null = quarantine_null_values(taxi_raw_data,taxi_raw_data.columns )
display(taxi_wo_null.limit(10))
display(taxi_q_null.limit(10))
print(f"Number of rows in taxi_wo_null: {taxi_wo_null.count()}")
print(f"Number of rows in taxi_q: {taxi_q_null.count()}")

In [0]:
taxi_wo_negative, taxi_q_negative = quarantine_negative_values(taxi_wo_null, [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee"
])
display(taxi_wo_negative.limit(10))
display(taxi_q_negative.limit(10))
print(f"Number of rows in taxi_wo_negative: {taxi_wo_negative.count()}")
print(f"Number of rows in taxi_q_negative: {taxi_q_negative.count()}")

In [0]:
taxi_valid_dates, taxi_date_invalid = quarantine_date_range(
    taxi_wo_null,
    start_date=_MIN_DATE,
    end_date=_MAX_DATE,
    date_format=DATE_FORMAT,
    cols=[
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime"
    ]
)
display(taxi_valid_dates.limit(10))
display(taxi_date_invalid.limit(10))
print(f"Number of rows in taxi_cleaned: {taxi_valid_dates.count()}")
print(f"Number of rows in taxi_q_date_invalid: {taxi_date_invalid.count()}")

## Load Weather Data

In [0]:
weather_raw_data = SparkRepository(spark).read(
    path="/Volumes/dbw_schwarz_test/default/temp/bronze/new_york_weather_daily_2026.csv",
    file_format="csv",
    header=True,
    schema=NEW_YORK_WEATHER_SCHEMA
)

print(f"Number of rows in weather_raw_data: {weather_raw_data.count()}")
display(weather_raw_data.limit(10))

In [0]:
weather_wo_null, weather_q_null = quarantine_null_values(weather_raw_data,weather_raw_data.columns )
display(weather_wo_null.limit(10))
display(weather_q_null.limit(10))
print(f"Number of rows in weather_wo_null: {weather_wo_null.count()}")
print(f"Number of rows in weather_q_null: {weather_q_null.count()}")

In [0]:
weather_wo_negative, weather_q_negative = quarantine_negative_values(weather_wo_null, [
    "rhum",
    "prcp",
    "wspd",
    "pres",
    "cldc"
])
display(weather_wo_negative.limit(10))
display(weather_q_negative.limit(10))
print(f"Number of rows in weather_wo_negative: {weather_wo_negative.count()}")
print(f"Number of rows in weather_q_negative: {weather_q_negative.count()}")

In [0]:

weather_q_negative = weather_q_negative.withColumn(
    "date",
    to_date(
        concat_ws("-", "year", "month", "day"),
        "yyyy-M-d"
    )
)